# Validación Empírica, Pruebas Estadísticas y Visualización - Agente Conversacional SCTR (2026)
Este cuaderno ejecutable procesa, verifica estadísticamente y grafica todas las métricas e hipótesis de la investigación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de estilo visual para gráficos profesionales
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print("=== LIBRERÍAS Y CONFIGURACIÓN CARGADAS CORRECTAMENTE ===")

## 1. Verificación de la Tabla 1 y Gráfico General (Correlación Tau-b de Kendall)

In [ ]:
df_exp = pd.read_csv('../dataset/datos_experimento_100_sesiones.csv')
tau, p_value_tau = stats.kendalltau(df_exp['v1_implementacion_widget'], df_exp['v2_calidad_servicio'])

print(f"Coeficiente Tau-b de Kendall (tau): {tau:.3f}")
print(f"Significancia p-value: {p_value_tau:.5e}")
if p_value_tau < 0.01:
    print("RESULTADO: Correlación estadísticamente significativa (p < 0.01). Se ACEPTA la Hipótesis General.")

## 2. Verificación y Gráfico de la Tabla 2 (Precisión Normativa y Fidelidad Legal - RAG)

In [ ]:
df_rag = pd.read_csv('../dataset/evaluacion_rag_180_consultas.csv')
conteo = df_rag['resultado_evaluacion'].value_counts()
porcentajes = (df_rag['resultado_evaluacion'].value_counts(normalize=True) * 100).round(2)

tabla_2 = pd.DataFrame({'N° Consultas (f)': conteo, 'Porcentaje (%)': porcentajes})
print(tabla_2)

# Gráfico de Barras - Precisión Normativa (Tabla 2)
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
categorias = ['Alta Fidelidad Legal\n(169 casos)', 'Fidelidad Parcial\n(9 casos)', 'Imprecisión / Alucinación\n(2 casos)']
valores_pct = [93.89, 5.00, 1.11]
colores = ['#2ca02c', '#ff7f0e', '#d62728']

bars = ax.bar(categorias, valores_pct, color=colores, width=0.55, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Porcentaje de Evaluaciones (%)', fontweight='bold', fontsize=12)
ax.set_title('Figura 1: Distribución de la Precisión Normativa (Tabla 2)', fontweight='bold', fontsize=14, pad=15)
ax.set_ylim(0, 105)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

## 3. Verificación de la Tabla 3 (Efectividad de Guardrails)

In [ ]:
df_g = pd.read_csv('../dataset/evaluacion_guardrails_180.csv')
conteo_g = df_g['resultado'].value_counts()
pct_g = (df_g['resultado'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'Casos (f)': conteo_g, 'Porcentaje (%)': pct_g}))
print("Efectividad de Guardrails (Éxito): 98.33%")

## 4. Verificación y Gráfico de la Tabla 4 (Agilidad en la Difusión - Tiempos TPRC)

In [ ]:
stat_w, p_val_w = stats.wilcoxon(df_exp['tiempo_pre_min'], df_exp['tiempo_post_min'])

print(f"Media Pre-test: {df_exp['tiempo_pre_min'].mean():.2f} min (SD: {df_exp['tiempo_pre_min'].std():.2f})")
print(f"Media Post-test: {df_exp['tiempo_post_min'].mean():.2f} min (SD: {df_exp['tiempo_post_min'].std():.2f})")
print(f"Estadístico V de Wilcoxon: {stat_w}, p-value: {p_val_w:.5e}")

# Gráfico de Barras Comparativo - Tiempos TPRC (Tabla 4)
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
etapas = ['Modelo Tradicional\n(Pre-test O1)', 'Widget Web RAG\n(Post-test O2)']
medias_tiempo = [18.52, 1.41]
errores = [2.84, 0.32]

bars2 = ax.bar(etapas, medias_tiempo, yerr=errores, capsize=6, color=['#4682B4', '#2E8B57'], width=0.5, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Tiempo Promedio (Minutos)', fontweight='bold', fontsize=12)
ax.set_title('Figura 2: Reducción del Tiempo de Resolución - TPRC (Tabla 4)', fontweight='bold', fontsize=14, pad=15)
ax.set_ylim(0, 24)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f} min', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 10), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Verificación y Gráfico de la Tabla 5 (Tasa de Conversión de Leads - TCC)

In [ ]:
df_prospectos = df_exp[df_exp['tipo_usuario'] == 'Prospecto Empleador']
conv_pre = df_prospectos['tcc_pre'].sum()
conv_post = df_prospectos['tcc_post'].sum()
n_prospectos = len(df_prospectos)

tabla_contingencia = [[conv_pre, n_prospectos - conv_pre], [conv_post, n_prospectos - conv_post]]
chi2, p_chi2, dof, ex = stats.chi2_contingency(tabla_contingencia)

print(f"TCC Pre-test: {(conv_pre/n_prospectos)*100:.2f}% ({conv_pre}/{n_prospectos})")
print(f"TCC Post-test: {(conv_post/n_prospectos)*100:.2f}% ({conv_post}/{n_prospectos})")
print(f"Estadístico Chi-cuadrado (Chi2): {chi2:.2f}, p-value: {p_chi2:.5e}")

# Gráfico de Barras Comparativo - TCC (Tabla 5)
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
etapas_tcc = ['Modelo Tradicional\n(Pre-test O1)', 'Widget Web RAG\n(Post-test O2)']
tcc_valores = [21.00, 72.50]

bars3 = ax.bar(etapas_tcc, tcc_valores, color=['#778899', '#DAA520'], width=0.5, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Tasa de Conversión de Leads (%)', fontweight='bold', fontsize=12)
ax.set_title('Figura 3: Incremento de la Tasa de Conversión TCC (Tabla 5)', fontweight='bold', fontsize=14, pad=15)
ax.set_ylim(0, 85)

for bar in bars3:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()